# Chapter 6a — Retrieval-Augmented Generation (Hotels)

Companion code for the **first half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

We connect the FAISS retrieval from Chapter 4 to the LLM prompting from Chapter 5 to produce a complete, grounded RAG pipeline:

1. **Retrieve** the most relevant hotel reviews for the user's query.
2. **Augment** the prompt with the retrieved context.
3. **Generate** a Markdown answer with inline citations.

We then graduate from FAISS (vectors live in process memory) to **Qdrant** — a real vector database that supports payloads and metadata filtering (e.g. *only Istanbul hotels*).

> Reusable code: `rag_pipeline.py`, `qdrant_helpers.py`.

## 0. Setup

Needs `OPEN_ROUTER_API_KEY` in your `.env` (sign up at https://openrouter.ai).

In [2]:
import sys, os
sys.path.insert(0, '.')
sys.path.insert(0, '../chapter_04_semantic_search')

import numpy as np
import torch
from IPython.display import Markdown, display

from data_loader import load_paris_reviews, load_hotel_reviews
from search import load_embedding_model, get_embeddings, build_faiss_cosine_index
from rag_pipeline import generate_answer, search_hotels_by_query
from qdrant_helpers import (
    make_in_memory_client,
    reset_collection,
    upsert_hotel_reviews,
    search_hotels,
)

## 1. Build the index (Paris-only, FAISS)

Same setup as Chapter 4 — we just keep the index alive so we can query it from RAG.

In [3]:
df_paris = load_paris_reviews()
print(f"Paris reviews: {len(df_paris):,}")

embed_model = load_embedding_model()
if torch.cuda.is_available(): embed_model = embed_model.to("cuda")
elif torch.backends.mps.is_available(): embed_model = embed_model.to("mps")

review_embeddings = embed_model.encode(df_paris["review_text"].tolist(), show_progress_bar=True).astype("float32")
faiss_index = build_faiss_cosine_index(review_embeddings)
print("Index ready.")

Paris reviews: 1,200


<All keys matched successfully>


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Index ready.


## 2. Retrieve only — what FAISS hands to the LLM

Before we generate, look at the raw retrieval. RAG quality is bounded above by retrieval quality.

In [4]:
query = "Hotel with a view of the Eiffel tower."
results = search_hotels_by_query(query, embed_model, faiss_index, df_paris, k=5)
for r in results:
    print(f"{r['rank']}. {r['hotel_name']}  (cos={r['cosine_similarity']:.3f})")
    print(f"   {r['review_text'][:160]}...\n")

1. Pullman Paris Eiffel Tower Hotel  (cos=0.823)
   Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hot...

2. Pullman Paris Eiffel Tower Hotel  (cos=0.808)
   If you stay at this hotel it is for the amazing views of the Eiffel Tower and for the pictures.  The photos and memories of being on the balcony looking at the ...

3. Citadines Tour Eiffel Paris  (cos=0.794)
   Had a Eiffel tower balcony view room and the view did not disappoint was absolutely amazing and so nice to see everyone from the room or balcony. Staff at recep...

4. Hotel Marignan Champs-Elysees  (cos=0.793)
   Very good hotel near the Champs-Elysees.  The superior rooms are very nice and a good size.  The Eiffel Tower suite has an excellent view of the Eiffel Tower. W...

5. Hotel Tourisme Avenue  (cos=0.791)
   Nice Hotel only a few minutes away from Eiffel tower. 3 Metro lines in Front of the hotel, supermar

## 3. Full RAG: retrieve → augment → generate

We stream the answer from an OpenRouter-hosted LLM (default: `qwen/qwen3-8b`).

In [5]:
query = "Hotels with a view of the Eiffel Tower"
answer, sources = generate_answer(query, embed_model, faiss_index, df_paris, k=10)

Pullman Paris Eiffel Tower Hotel is highly recommended for its exceptional view of the Eiffel Tower, with reviews highlighting stunning balcony vistas and proximity to the landmark [1][2]. The hotel offers spacious, luxurious rooms and suites, including a top-floor option with a balcony, ensuring an unforgettable experience [1]. Another top choice is Hotel Marignan Champs-Elysees, which features an Eiffel Tower suite with an excellent view, complemented by friendly staff and a convenient location [3]. Citadines Tour Eiffel Paris provides affordable apartments with direct Eiffel Tower views, though rooms are described as cozy rather than luxurious [5][6]. The Cler Hotel also offers an Eiffel Tower view in specific rooms, though other rooms provide street-level vistas [9]. For budget-friendly options, Hotel Tourisme Avenue is within walking distance of the Eiffel Tower, though its reviews do not explicitly mention tower views [4]. All these hotels emphasize convenience and scenic experie

## 4. Spread to all cities — and add metadata filtering with Qdrant

FAISS gave us fast vector search. But it doesn't know that *some hotels are in Istanbul, others in Paris*. To filter by city we'd have to keep parallel arrays in sync ourselves.

**Qdrant** stores `{vector, payload}` per point and supports server-side filters. That's the production shape.

In [6]:
# Load all cities, not just Paris
df_all = load_hotel_reviews().drop_duplicates().reset_index(drop=True)
df_all = df_all.dropna(subset=["review_text"]).reset_index(drop=True)
print(f"All-city reviews: {len(df_all):,}")
print(df_all.locality.value_counts().head(10))

All-city reviews: 4,867
locality
London           1200
Paris            1200
New York City    1197
San Francisco     680
Istanbul          590
Name: count, dtype: int64


In [7]:
# Embed (this takes a while; restrict for a quick demo)
DEMO_LIMIT = 5000  # set None to embed everything
df_demo = df_all.head(DEMO_LIMIT).reset_index(drop=True) if DEMO_LIMIT else df_all
all_embeddings = embed_model.encode(df_demo["review_text"].tolist(), show_progress_bar=True).astype("float32")
print(all_embeddings.shape)

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

(4867, 768)


In [8]:
qdrant = make_in_memory_client()
COLLECTION = "hotel_reviews"
reset_collection(qdrant, COLLECTION, vector_size=all_embeddings.shape[1])
upsert_hotel_reviews(qdrant, COLLECTION, df_demo, all_embeddings)
print(f"Upserted {len(df_demo):,} reviews into '{COLLECTION}'.")

Upserted 4,867 reviews into 'hotel_reviews'.


### City-filtered semantic search

In [9]:
query = "Amazing hotel close to everything"
city_filter = "Istanbul"

hits = search_hotels(query, embed_model, qdrant, collection_name=COLLECTION, city=city_filter, k=5)
for i, h in enumerate(hits, 1):
    print(f"{i}. {h.payload['hotel_name']}  ({h.payload['locality']})  score={h.score:.3f}")
    print(f"   {h.payload['review_text'][:160]}...\n")

1. White House Hotel Istanbul  (Istanbul)  score=0.718
     One of the best experiences I’ve had at a hotel before. The hotel is amazing, highly beautiful and unique. I picked this hotel as boutique choice and it did n...

2. Skalion Hotel & Spa  (Istanbul)  score=0.703
   Really loved this hotel, every main attraction (main bazaar, hagia Sofia, blue mosque...) is on walking distance!! The staff was really really kind expecially N...

3. Pera Palace Hotel  (Istanbul)  score=0.699
   Amazing place in the world I am in love with this hotel amazing I wish i could live there for all my life best place in the world room service is amazing reserv...

4. Hotel Amira Istanbul  (Istanbul)  score=0.681
   Amazing 2 nights as an extended stopover. Hotel was perfectly located near great attractions. Front desk helped outline a great itinerary for us around the city...

5. Tomtom Suites  (Istanbul)  score=0.680
   Very good. I recommend for everybody this hotel. It is a 5* experience from design to

### Full RAG with city filter

Same retrieve → augment → generate flow, but the retrieval is now scoped to one city.

In [10]:
from rag_pipeline import get_openrouter_client

def generate_answer_qdrant(query, city=None, k=15, llm_model="qwen/qwen3-8b"):
    hits = search_hotels(query, embed_model, qdrant, collection_name=COLLECTION, city=city, k=k)
    context = "\n".join(
        f"Source {i+1}: {h.payload['hotel_name']} ({h.payload['locality']}) — score {h.score:.3f}\n"
        f"  Review: {h.payload['review_text']}"
        for i, h in enumerate(hits)
    )
    prompt = f'''Answer the user's query in Markdown using ONLY the retrieved sources below.
Cite inline as [1][2]. Be concise and concrete; no salutations.

Query: "{query}"
City filter: {city or "(none)"}

Sources:
{context}
'''
    client = get_openrouter_client()
    stream = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    out = ""
    for chunk in stream:
        d = chunk.choices[0].delta.content
        if d:
            print(d, end="", flush=True)
            out += d
    print()
    return out, hits

In [11]:
answer, sources = generate_answer_qdrant("Amazing hotel close to everything", city="Istanbul", k=10)

- **White House Hotel Istanbul** (Score: 0.718) [1] — Ample walking distance to main attractions, luxurious rooms, and a standout breakfast. Staff are welcoming, and the location offers easy access to restaurants and sights.  
- **Skalion Hotel & Spa** (Score: 0.703) [2] — Walking distance to Hagia Sophia, Blue Mosque, and the Grand Bazaar. Attached markets, pharmacies, and restaurants eliminate the need for taxis. Staff praised for kindness.  
- **Hotel Amira Istanbul** (Score: 0.681) [4] — Perfectly located near attractions; front desk provided a detailed itinerary. Breakfast featured fresh, custom bread. Guests returned for the experience.  
- **Tomtom Suites** (Score: 0.680) [5] — Boutique hotel with a Michelin-starred restaurant on-site. Staff organized transport and offered excellent dining recommendations. Rooms are spacious and well-designed.  
- **Pell Palace Hotel Spa** (Score: 0.679) [8] — Close to shops and attractions, with a friendly staff and a notable spa. Guests plan t

## What's next

Notebook **6b** repeats this pipeline on a very different corpus — research-paper abstracts — adding:

- Document chunking (papers can be longer than the embedding model's context window)
- HuggingFace `transformers` directly (skipping `sentence-transformers`)
- Persisting the Qdrant collection to disk